# Calcular entropies per a tots els datasets i guardar-les en un nou CSV

In [1]:
import pandas as pd

df = pd.read_csv("comparative_final_results.csv")

df.head()


,dataset_id,dataset_label,algorithm,original_size_bytes,compressed_size_bytes,compression_ratio,compression_percent,bits_per_base_total,bits_per_base_seq,size_header,...,decompress_exit_code,stderr_log,status,num_bases,num_A,num_C,num_G,num_T,num_N,num_unknown
0,bacillus_subtilis,bacteria,gzip,4275901,1300457,3.287999,69.586363,2.433091,2.467891,72,...,0,/home/helen/genomic_benchmark/results/2026_05_...,ok,4215606,1188073,919284,915112,1193137,0,0
1,bacillus_subtilis,bacteria,bzip2,4275901,1213681,3.523085,71.615783,2.270737,2.303215,72,...,0,/home/helen/genomic_benchmark/results/2026_05_...,ok,4215606,1188073,919284,915112,1193137,0,0
2,bacillus_subtilis,bacteria,genozip,4275901,1032858,4.139873,75.844670,1.932426,1.960066,72,...,0,/home/helen/genomic_benchmark/results/2026_05_...,ok,4215606,1188073,919284,915112,1193137,0,0
3,bacillus_subtilis,bacteria,mfcompress,4275901,1012990,4.221069,76.309321,1.895254,1.922362,72,...,0,/home/helen/genomic_benchmark/results/2026_05_...,ok,4215606,1188073,919284,915112,1193137,0,0
4,bacillus_subtilis,bacteria,naf,4275901,1103012,3.876568,74.203986,2.063681,2.093198,72,...,0,/home/helen/genomic_benchmark/results/2026_05_...,ok,4215606,1188073,919284,915112,1193137,0,0


In [2]:
print(df.columns)


Index(['dataset_id', 'dataset_label', 'algorithm', 'original_size_bytes',
       'compressed_size_bytes', 'compression_ratio', 'compression_percent',
       'bits_per_base_total', 'bits_per_base_seq', 'size_header',
       'entropy_acgtn_bits_per_base', 'entropy_acgt_bits_per_base',
       'compress_seconds', 'decompress_seconds', 'compress_MBps',
       'decompress_MBps', 'bitwise_ok', 'sequence_ok', 'sequence_ok_noN',
       'compress_exit_code', 'decompress_exit_code', 'stderr_log', 'status',
       'num_bases', 'num_A', 'num_C', 'num_G', 'num_T', 'num_N',
       'num_unknown'],
      dtype='object')


In [5]:
# columnes datasets

cols_datasets = ["dataset_id", "dataset_label", "original_size_bytes", "size_header", "num_bases", "num_A", "num_C", "num_G", "num_T", "num_N", "num_unknown"]
cols_algorithms = ["algorithm", "compressed_size_bytes", "compression_ratio", "compression_percent", "bits_per_base_seq", "compress_seconds", "decompress_seconds", "compress_MBps", "decompress_MBps", "bitwise_ok", "sequence_ok", "sequence_ok_noN", "compress_exit_code", "decompress_exit_code", "stderr_log", "status"]

total_cols = cols_datasets + cols_algorithms

print("Hi ha columnes duplicades a total_cols:", len(set(total_cols)) != len(total_cols))
print("Columnes esperades que falten:", [col for col in total_cols if col not in df.columns])
print("Columnes del DataFrame que no estan classificades:", [col for col in df.columns if col not in total_cols])

falten =  ["entropy_acgt_bits_per_base", "entropy_acgtn_bits_per_base", "entropy_seq_all_bits_per_base", "entropy_seq_all_headers_bits_per_base"]

Hi ha columnes duplicades a total_cols: False
Columnes esperades que falten: []
Columnes del DataFrame que no estan classificades: ['bits_per_base_total', 'entropy_acgtn_bits_per_base', 'entropy_acgt_bits_per_base']


In [14]:
#from comparative.entities import Dataset
# force pwd to be the root of the project
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))
print(Path("..").resolve())

from typing import List
print(f"Working directory: {Path.cwd()}")
from comparative.entities import Dataset
from comparative.datasets import load_datasets

datasets = load_datasets(Path("../DATA/data"))

/home/helen/genomic_benchmark
Working directory: /home/helen/genomic_benchmark/proves
Loading datasets from ../DATA/data...


In [15]:
df = df[total_cols]

In [16]:
entropy_rows = []

for ds in datasets:
    entropy_rows.append({
        "dataset_id": ds.dataset_id,
        "entropy_acgt_bits_per_base": ds.entropy_acgt_bits_per_base,
        "entropy_acgtn_bits_per_base": ds.entropy_acgtn_bits_per_base,
        "entropy_seq_all_bits_per_base": ds.entropy_seq_all_bits_per_base,
        "entropy_seq_all_headers_bits_per_base": ds.entropy_seq_all_headers_bits_per_base,
    })

entropy_df = pd.DataFrame(entropy_rows)

In [17]:
df = df.merge(
    entropy_df,
    on="dataset_id",
    how="left"
)

In [18]:
df.to_csv("comparative_results_updated.csv", index=False)